<a href="https://colab.research.google.com/github/xc308/Dataset_preparation_Fine_Tuning/blob/main/Full_Parameter_Fine_Tuning_of_Gemma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Aim:

    - Attempt to perform full-parameter fine-tuning of a pretrained Gemma model

    -  full-parameter fine-tuning, will encounter a real-world constraint: memory limitations.

In [1]:
# Install the custom package for this course.
!pip install "git+https://github.com/google-deepmind/ai-foundations.git@main"

import os

# For loading the Kaggle username and key.
from google.colab import userdata

# Load the Kaggle username and key.
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

os.environ["KERAS_BACKEND"] = "jax" # Set the Keras backend to JAX.

import keras # For training the model.
import keras_hub # For loading Gemma 3.
import pandas as pd # For loading the dataset.
from textwrap import fill # For making paragraphs more readable.
# For loading the formatting function from the lab
# "Format Text for Turn-Based Dialogue."
from ai_foundations import formatting

# Avoid memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
keras.utils.set_random_seed(812) # For making the training reproducible.

  Cloning https://github.com/google-deepmind/ai-foundations.git (to revision main) to /tmp/pip-req-build-kejhqhc6
  Running command git clone --filter=blob:none --quiet https://github.com/google-deepmind/ai-foundations.git /tmp/pip-req-build-kejhqhc6
  Resolved https://github.com/google-deepmind/ai-foundations.git to commit 524d6114bbce631dafc00ba3496607a0bc60c804
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


**Data pre-processing**


- fine-tuning data uses the tokenizer and the special delimiter tokens that were used during pre-training


- using the format that instruction-tuned Gemma models use
    - uses special tokens to mark the beginning and end of a turn, and indicates the role (user or model)

    - A single training example will be structured as follows:

<start_of_turn>user

What is Jollof rice?<end_of_turn>

<start_of_turn>model

Category: Food

Jollof rice is a popular and iconic one-pot rice dish that is a staple in many West African countries.<end_of_turn>




In [2]:
# Load the question-answer dataset.
africa_galore_qa = pd.read_json(
    "https://storage.googleapis.com/dm-educational/assets/ai_foundations/africa_galore_qa_v2.json"
)

questions = []  # List of formatted questions.
answers = []  # List of formatted answers.

for idx, row in africa_galore_qa.iterrows():
    # Run the format_qa function from the previous lab to format the question
    # and the answer.
    question, answer = formatting.format_qa(row)
    questions.append(question)
    answers.append(answer)



In [3]:
# Show the first set of inputs and outputs.
print(questions[0])

<start_of_turn>user
What is Kente Cloth?<end_of_turn>



In [4]:
print(answers[0])

<start_of_turn>model
Category: Textile
The vibrant colors and intricate patterns of Kente cloth, a symbol of Ghanaian royalty and prestige, tell stories of history, culture, and social status. Woven on narrow looms by skilled artisans, each strip of Kente is a testament to patience and artistry. The geometric designs, rich with symbolism, represent proverbs, historical events, and important figures. Worn during special occasions and ceremonies, Kente cloth embodies the spirit of Ghana, its vibrant culture, and its rich history. From the bright yellows and golds representing royalty to the deep blues and greens symbolizing spirituality, Kente is a visual language, a wearable expression of Ghanaian identity and heritage.<end_of_turn>


In [5]:
print(fill(answers[0], replace_whitespace=False))

<start_of_turn>model
Category: Textile
The vibrant colors and
intricate patterns of Kente cloth, a symbol of Ghanaian royalty and
prestige, tell stories of history, culture, and social status. Woven
on narrow looms by skilled artisans, each strip of Kente is a
testament to patience and artistry. The geometric designs, rich with
symbolism, represent proverbs, historical events, and important
figures. Worn during special occasions and ceremonies, Kente cloth
embodies the spirit of Ghana, its vibrant culture, and its rich
history. From the bright yellows and golds representing royalty to the
deep blues and greens symbolizing spirituality, Kente is a visual
language, a wearable expression of Ghanaian identity and
heritage.<end_of_turn>


In the case of the Keras implementation of the Gemma model, you only need to specify the prompt and what you would like the model to generate. It then automatically constructs tokenized examples that can be used to fine-tune a Gemma language model.

In the background, it concatenates the prompts and responses and excludes the tokens in the prompt from the loss computation.

Note, however, that it does not automatically add <start_of_turn> or <end_of_turn> tokens, so you still have to provide these tokens to the model, as you did in the previous cell.

**Preparing data for the Keras model**

The Keras implementation of Gemma expects the fine-tuning data to be structured as a Python dictionary with two specific keys:

    - prompts for the prompts ( questions in the case of the flashcard generator) and


    - responses for the generations that the model should output (the answers in the case of the flashcard generator).



In [6]:
data = {
    "prompts": questions,
    "responses": answers
}

**Loading the Gemma model with Keras**

- loads the pre-trained Gemma-1B model using Keras

- this model has been trained on the next-word prediction task and it has not been optimized for answering questions.

-

In [7]:
# Load the Gemma3-1B Keras model.
model = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_1b")
model.summary()

# Set the maximum sequence length of the model for padding and batching.
model.preprocessor.sequence_length = 400

Preprocessor: "gemma3_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer (Gemma3Tokenizer)                            │                      Vocab size: 262,144 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma3_backbone               │ (None, None, 1152)        │     999,885,952 │ padding_mask[0][0],        │
│ (Gemma3Backbone)              │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 262144)      │     301,989,888 │ gemma3_backbone[0][0]      │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 999,885,952 (3.72 GB)

 Trainable params: 999,885,952 (3.72 GB)

 Non-trainable params: 0 (0.00 B)

**Understanding Gemma's Memory Footprint**


- Gemma 1B model has approximately 1.3 billion parameters (1B for the main transformer blocks and 300M for the token embeddings)

- A model of this size typically requires around 4GB of memory, so it fits on a T4 GPU with 15GB of memory

- a 27B model, that is a model with around 27 billion parameters would require roughly 100 GB of memory (27  ×  ~3.7 GB), which exceeds the capacity of a single T4 GPU multiple times.

**Prompting the base model**

-

In [8]:
prompt = "What is Jollof rice?"
response = model.generate(prompt, max_length=200)
print(fill(response, replace_whitespace=False))

What is Jollof rice?

Jollof rice is a popular dish in West Africa,
especially in Nigeria. It is made from rice, tomatoes, onions, and
other spices. The rice is cooked in a pot with a lot of water and
spices, and then served with meat or fish. Jollof rice is a delicious
and filling dish that is perfect for any meal.

What is the history of
Jollof rice?

Jollof rice is a dish that has been around for a long
time. It is believed to have originated in West Africa, but it has
since spread to other parts of the world. The dish is often associated
with the culture and traditions of West Africa, and it is a popular
dish in many countries around the world.

What are the ingredients in
Jollof rice?

Jollof rice is made from rice, tomatoes, onions, and
other spices. The rice is cooked in a pot with a lot of water and
spices,


**Attempting full-parameter fine-tuning**

- Fine-tuning requires significantly more memory than using a model for inference. For each parameter, the training process must store not only the current value of the parameter itself, but also the gradient and other variables used by the optimizer. Because of this, training a model requires several times more memory than using a model for generations.


-

In [9]:
model.fit(data, epochs=1, batch_size=32, verbose=1)

XlaRuntimeError: INTERNAL: No reference output found!